*GeoPlants Dataset*

## Load Dataset

In [1]:
## Import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### Import the created function
from src.utils import merge_metadata_data

- ATENTION

Note that only one path is given for the df containing training data. That is due on the notebook "ecoregions" where the GRECO (Grande Regions Ecologique) was used to interpolate and extracted two more columns at the original dataset. These columns are: GRECO, codeser. Where GRECO is the higher level aggregation and codeser is the so called "sylvioecoregions" which are uniformly representative ecological regions containing agro-forest (silvo pastoril) systems and forest interaction. 

See more at the notebook.

On this case, the path names where changed and now only the train data is called, being stored inside the path: output/ecoregions/train_join.parquet

In [38]:
## Path names
path_train = "/mnt/d/desktop/Copernicus/classes/3-semester/ml/species/dataset/train_join_agg.parquet"
path_test = ""

In [39]:
## Standard Training dataset load 
df = pd.read_parquet(path_train, engine='fastparquet')
df.set_index("surveyId",inplace=True)

## Select the X train dataset
## Drop non feature columns but preserve the df to map easily
## since the order to the X index is the same order of the df.
columns_to_drop = ['lon','lat','speciesId','predict','GRECO','codeser','index']

X = df.drop(columns=columns_to_drop)

y = df[['predict']]

## Pre Processing

### Data Cleaning

In [ ]:
import re

hf_cols = ['footprint_HumanFootprint-Built1994','footprint_HumanFootprint-Built2009',
       'footprint_HumanFootprint-croplands1992','footprint_HumanFootprint-croplands2005',
       'footprint_HumanFootprint-Lights1994',  'footprint_HumanFootprint-Lights2009',
       'footprint_HumanFootprint-NavWater1994', 'footprint_HumanFootprint-NavWater2009',
       'footprint_HumanFootprint-Pasture1993',  'footprint_HumanFootprint-Pasture2009',
       'footprint_HumanFootprint-Popdensity1990', 'footprint_HumanFootprint-Popdensity2010',
       'footprint_HumanFootprint-Railways', 'footprint_HumanFootprint-Roads',
       'footprint_HumanFootprint-HFP1993', 'footprint_HumanFootprint-HFP2009']

## Stardadization of data cleaning
## Create a safe copy
Xx = X[hf_cols].copy()

## Replace Wrong Variables present in the HumanFootprint - See section HumanFootprint to understand the why these values are wrong
## Get all columns with value lesser than zero
cols_less_zero = Xx.columns[(Xx<0).any(axis=0)]
print(cols_less_zero)
Xx = Xx.where(Xx > 0, 0.0, axis='index') #replace by zero

## Replace all values above a given threshold
## e.g HF Built should not be bigger than 100
col_to_clean = ['footprint_HumanFootprint-Built1994','footprint_HumanFootprint-Built2009',
                'footprint_HumanFootprint-croplands1992','footprint_HumanFootprint-croplands2005',
                'footprint_HumanFootprint-Lights1994','footprint_HumanFootprint-Lights2009',
                'footprint_HumanFootprint-Pasture1993','footprint_HumanFootprint-Pasture2009']
Xx[col_to_clean] = Xx[col_to_clean].where(Xx[col_to_clean] <= 100, Xx[col_to_clean].median(), axis='index') ## replace by the median 

## Replace HPF by the median
col_to_clean =['footprint_HumanFootprint-HFP1993', 'footprint_HumanFootprint-HFP2009']
Xx[col_to_clean] = Xx[col_to_clean].where(Xx[col_to_clean]<51, Xx[col_to_clean].median(), axis='index')

## Replace a likely wrong value in Pop Density 
## It just transfer the value of pop density to the next/before pop density year - 97% percentile is 10.0, which means that value above are likely wrong
col90 = ['footprint_HumanFootprint-Popdensity1990']
col2010 = ['footprint_HumanFootprint-Popdensity2010']

print(f" The 96% percentile of the Pop2010: {np.percentile(Xx[col2010],98)}")
print(f" The 96% percentile of the Pop1990: {np.percentile(Xx[col90],98)}")
print(f" Max value Pop2010: {Xx[col2010].max().values}")
print(f" Max value Pop1990: {Xx[col90].max().values}")
#Xx[col90] = Xx[col90].where(Xx[col90]< 11, Xx[col2010], axis='index')
## Replace values of Pop2010 with the same of Pop90
Xx[col2010] = Xx[col2010].where(Xx[col2010]< 11, Xx[col90], axis='index')


# ### CONSTRUCT NEW COLUMNS
## Find columns with year in the name of the column
hf_cols_temp =[]
for i in range(0,len(hf_cols)):
    find_number = re.findall(r'\d{4}$', hf_cols[i].split("-")[-1])
    #print(i,find_number)
    if find_number:
        hf_cols_temp.append(hf_cols[i])     
print(f"Temporal columns: {hf_cols_temp}")  

# ## Create new columns that contains the difference between two years
new_columns=[]
for i in range(0,len(hf_cols_temp),2):
    new_name = hf_cols_temp[i].split('-')[-1]
    name = re.findall(r'^(.+?)\d+$',new_name)[0]
    year0 = re.findall(r'\d{4}$', new_name)[0]
    year1 = re.findall(r'\d{4}$', hf_cols_temp[i+1])[0]
    full_name= f"{name}-{year0}-{year1}"
    new_columns.append(full_name)
    Xx[full_name] = Xx[hf_cols_temp[i+1]] - Xx[hf_cols_temp[i]]  

print(f"Newly created columns: {new_columns}")


## Plot the distribution 
# fig, ax = plt.subplots(figsize=(7,5))
# ax = sns.boxplot(data=Xx)
# plt.xticks(rotation=45, ha='right')
# ax.set_title('Human Footprint - New Variables')
# ax.set_xlabel('')  # Remove x-axis label if needed
# plt.tight_layout()  # Prevents labels from being cut off
# plt.show()


## It needs to replace all the variables RailWay bigger than 10 to 0. They are probably wrong
Xx['footprint_HumanFootprint-Railways'] = Xx['footprint_HumanFootprint-Railways'].where(Xx['footprint_HumanFootprint-Railways']< 10, 0.0, axis='index')

## Clip Values of Cropland bigger than threshold and light lesser than threhold
Xx['croplands-1992-2005'] = Xx['croplands-1992-2005'].clip(-10,20)
Xx['Lights-1994-2009'] = Xx['Lights-1994-2009'].clip(-8,7)

# #### -------------------------------------------------
## --------------------- CONCAT ---------------------------------------
list_hf_railways_roads = [col for col in hf_cols if col not in hf_cols_temp]

## drop if they still exists:
if Xx.columns.isin(hf_cols_temp).any()==True:
    Xx = Xx.drop(columns=hf_cols_temp, axis=1)

## Apply a clip value 
X = pd.concat([X.drop(columns=hf_cols), Xx], axis=1)

#### -----------------------------
### Prepare Climatic Variables 
### SCALE AND OFFSET 
print('---'*30)
print('Climatic Variables are being scaled and offset.')
print()
climatic_variables = ['average_Bio1', 'average_Bio2', 'average_Bio3', 'average_Bio4',
       'average_Bio5', 'average_Bio6', 'average_Bio7', 'average_Bio8',
       'average_Bio9', 'average_Bio10', 'average_Bio11', 'average_Bio12',
       'average_Bio13', 'average_Bio14', 'average_Bio15', 'average_Bio16',
       'average_Bio17', 'average_Bio18', 'average_Bio19']

scale = 0.1
offset_vector= (-273.15,0,0,0,-273.15,-273.15,0,-273.15,-273.15,-273.15,-273.15,0,0,0,0,0,0,0,0)

X[climatic_variables] = X[climatic_variables]*0.1 + offset_vector


########### ------------------------------  FILL
## Deals with missing values
print(f"----"*30)
print(f"Filling Nan values.")


cat_cols = ["landcover_LandCover"]
not_cat_cols = [col for col in X.columns if col not in cat_cols]

print(f"Missing values:{X.isna().sum().sum()}")

## fill na values with median
X[not_cat_cols] = X[not_cat_cols].fillna(X.median(numeric_only=True))

### check for categorical columns, if yes, it will be filled them by mode
X[cat_cols] = X[cat_cols].fillna(X[cat_cols].mode())

print(f"Missing values after imputation:{X.isna().sum().sum()}")




Index(['footprint_HumanFootprint-NavWater1994', 'footprint_HumanFootprint-NavWater2009'], dtype='object')
 The 96% percentile of the Pop2010: 2147483647.0
 The 96% percentile of the Pop1990: 127.0
 Max value Pop2010: [2.14748365e+09]
 Max value Pop1990: [127.]
Temporal columns: ['footprint_HumanFootprint-Built1994', 'footprint_HumanFootprint-Built2009', 'footprint_HumanFootprint-croplands1992', 'footprint_HumanFootprint-croplands2005', 'footprint_HumanFootprint-Lights1994', 'footprint_HumanFootprint-Lights2009', 'footprint_HumanFootprint-NavWater1994', 'footprint_HumanFootprint-NavWater2009', 'footprint_HumanFootprint-Pasture1993', 'footprint_HumanFootprint-Pasture2009', 'footprint_HumanFootprint-Popdensity1990', 'footprint_HumanFootprint-Popdensity2010', 'footprint_HumanFootprint-HFP1993', 'footprint_HumanFootprint-HFP2009']
Newly created columns: ['Built-1994-2009', 'croplands-1992-2005', 'Lights-1994-2009', 'NavWater-1994-2009', 'Pasture-1993-2009', 'Popdensity-1990-2010', 'HFP-1993

### Pre-Processing

- StandarScaler
- OneHotEncoder

In [43]:
## Name cols
numeric_features = not_cat_cols
categorical_features = cat_cols 

## Numeric features
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")),
           ("scaler", StandardScaler())]
)


## Categorical Features
categorical_transformer = Pipeline(
                steps=[
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ##("selector", SelectPercentile(chi2, percentile=50)),
                ]
)

## Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [ ]:
# Convert to DataFrame
X_transformed_df = pd.DataFrame(preprocessor.fit_transform(X),
                                columns=preprocessor.get_feature_names_out(),
                                index=X.index)


In [45]:
X_transformed_df

,num__average_Bio1,num__average_Bio2,num__average_Bio3,num__average_Bio4,num__average_Bio5,num__average_Bio6,num__average_Bio7,num__average_Bio8,num__average_Bio9,num__average_Bio10,...,cat__landcover_LandCover_5.0,cat__landcover_LandCover_8.0,cat__landcover_LandCover_9.0,cat__landcover_LandCover_10.0,cat__landcover_LandCover_11.0,cat__landcover_LandCover_12.0,cat__landcover_LandCover_13.0,cat__landcover_LandCover_14.0,cat__landcover_LandCover_16.0,cat__landcover_LandCover_17.0
surveyId,,,,,,,,,,,,,,,,,,,,,
333,0.347195,0.260223,0.401941,-0.783924,0.240344,0.530852,-0.408877,0.014173,0.601191,0.108783,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
410,-0.137349,0.332205,0.401941,1.111207,0.167383,-0.400396,0.660988,-0.551047,-1.425111,0.108783,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
779,0.011741,0.116260,0.401941,-0.015829,0.057941,0.170369,-0.150633,-0.345513,-0.992833,-0.005940,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1218,-0.137349,-0.099684,0.401941,0.066002,-0.124463,0.020168,-0.150633,-0.448280,-1.046868,-0.120663,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015,0.719922,0.836075,0.401941,1.352980,1.517168,0.140329,1.361934,1.118923,0.979434,1.179532,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3917273,-1.218257,0.188242,0.401941,-0.216687,-1.218883,-0.911081,-0.113742,1.195998,-1.722302,-1.306135,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3917646,1.539921,-0.027703,0.401941,0.246402,1.736052,1.131658,0.365853,0.784929,1.155046,1.676665,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3917824,-0.100077,-0.027703,0.401941,0.039965,-0.124463,0.080248,-0.224417,-0.396896,0.533647,-0.082422,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Cross-Validation Strategy

In [48]:
from sklearn.model_selection import GroupKFold

In [ ]:
print(df.shape, X.shape)

kfold = GroupKFold(n_splits=4)
group_kfolds = kfold.get_n_splits(X,y,groups = df['GRECO'])


dict_fold = {}
for i, (train_index, test_index) in enumerate(kfold.split(X, y, df['GRECO'])):
    dict_fold[i] = {
        'train': train_index,
        'test': test_index
    }
    

In [51]:
dict_fold

{0: {'train': array([    0,     1,     2, ..., 13496, 13498, 13500], shape=(9778,)),
  'test': array([    4,    19,    25, ..., 13494, 13497, 13499], shape=(3723,))},
 1: {'train': array([    0,     1,     4, ..., 13496, 13497, 13499], shape=(10109,)),
  'test': array([    2,     3,     7, ..., 13493, 13498, 13500], shape=(3392,))},
 2: {'train': array([    0,     1,     2, ..., 13498, 13499, 13500], shape=(10127,)),
  'test': array([    5,     6,     8, ..., 13489, 13495, 13496], shape=(3374,))},
 3: {'train': array([    2,     3,     4, ..., 13498, 13499, 13500], shape=(10489,)),
  'test': array([    0,     1,    10, ..., 13465, 13467, 13477], shape=(3012,))}}

## Feature Exploration

### Model Suit

In [ ]:
pipe = colu

## 